In [3]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/heatwave_corpus_final.csv")

print("File found:", data_path.exists())
print("File location:", data_path.resolve())

corpus = pd.read_csv(
    data_path,
    encoding="utf-8-sig"
)

corpus_analysis = corpus[
    corpus["include_in_final_analysis"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
].copy()

print("Stored records:", len(corpus))
print("Included documents:", len(corpus_analysis))

File found: True
File location: C:\Users\mishu\Desktop\heatwave_project\data\heatwave_corpus_final.csv
Stored records: 80
Included documents: 72


In [4]:
##token analysis

import re

token_pattern = r"\b[\w]+(?:['’\-][\w]+)*\b"

tokens_unfiltered = (
    corpus_analysis[
        ["doc_id", "source_type", "publication_year", "clean_text"]
    ]
    .assign(
        token=lambda x:
        x["clean_text"].str.lower().str.findall(token_pattern)
    )
    .explode("token")
    .dropna(subset=["token"])
    .drop(columns="clean_text")
    .reset_index(drop=True)
)

print("Total tokens:", len(tokens_unfiltered))
print("Unique tokens:", tokens_unfiltered["token"].nunique())

Total tokens: 94940
Unique tokens: 7371


In [5]:
top_tokens = (
    tokens_unfiltered["token"]
    .value_counts()
    .head(40)
    .rename_axis("token")
    .reset_index(name="frequency")
)

top_tokens

,token,frequency
0,the,4250
1,and,3428
2,of,2672
3,to,2401
4,in,2094
5,a,1179
6,heat,1165
7,for,869
8,are,831
9,is,820


In [6]:
##removing stopwords

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = {"et", "al"}

tokens_filtered = tokens_unfiltered[
    ~tokens_unfiltered["token"].isin(ENGLISH_STOP_WORDS)
    & ~tokens_unfiltered["token"].isin(custom_stopwords)
    & tokens_unfiltered["token"].str.contains(r"[a-z]", regex=True)
].copy()

filtered_top_tokens = (
    tokens_filtered["token"]
    .value_counts()
    .head(40)
    .rename_axis("token")
    .reset_index(name="frequency")
)

print("Tokens before filtering:", len(tokens_unfiltered))
print("Tokens after filtering:", len(tokens_filtered))

filtered_top_tokens

Tokens before filtering: 94940
Tokens after filtering: 55557


,token,frequency
0,heat,1165
1,health,792
2,children,720
3,climate,671
4,bangladesh,575
5,extreme,487
6,change,412
7,temperatures,405
8,water,289
9,exposure,251


In [7]:
## TF-IDF
##

number_of_documents = corpus_analysis["doc_id"].nunique()

token_statistics = (
    tokens_filtered
    .groupby("token")
    .agg(
        frequency=("token", "size"),
        document_frequency=("doc_id", "nunique")
    )
    .reset_index()
)

token_statistics["document_percentage"] = (
    token_statistics["document_frequency"]
    / number_of_documents
    * 100
).round(1)

token_statistics = token_statistics.sort_values(
    ["document_frequency", "frequency"],
    ascending=False
).reset_index(drop=True)

token_statistics.head(40)

,token,frequency,document_frequency,document_percentage
0,heat,1165,69,95.8
1,bangladesh,575,68,94.4
2,climate,671,67,93.1
3,children,720,65,90.3
4,health,792,64,88.9
5,temperatures,405,62,86.1
6,change,412,60,83.3
7,vulnerable,182,59,81.9
8,extreme,487,58,80.6
9,risk,161,54,75.0


In [8]:
#further cleaning 

bangladesh_scope_exclusions = [
    "DOC_045",
    "DOC_050",
    "DOC_065",
    "DOC_066",
    "DOC_072",
    "DOC_076",
    "DOC_078"
]
##excluded because these articles are not related to Bangladesh

# Preserve the previous inclusion decision
previously_included = (
    corpus["include_in_clean_analysis"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# Create the final inclusion variable
corpus["include_in_final_analysis"] = (
    previously_included
    & ~corpus["doc_id"].isin(bangladesh_scope_exclusions)
)

# Record exclusion reasons
corpus["final_exclusion_reason"] = ""

corpus.loc[
    ~previously_included,
    "final_exclusion_reason"
] = "Previously excluded as an exact duplicate"

corpus.loc[
    corpus["doc_id"].isin(bangladesh_scope_exclusions),
    "final_exclusion_reason"
] = (
    "Background source: discusses children and heat globally "
    "but has no substantive Bangladesh focus"
)

# Recreate the analytical corpus
corpus_analysis = corpus[
    corpus["include_in_final_analysis"]
].copy()

print("All stored records:", len(corpus))
print("Final included documents:", len(corpus_analysis))
print(corpus_analysis["source_type"].value_counts())

All stored records: 80
Final included documents: 72
source_type
news paper       46
civil society    16
scientific        9
others            1
Name: count, dtype: int64


In [9]:
corpus.to_csv(
    "../data/heatwave_corpus_final.csv",
    index=False,
    encoding="utf-8-sig"
)

In [10]:
##token analysis 2

import re

token_pattern = r"\b[\w]+(?:['’\-][\w]+)*\b"

tokens_unfiltered = (
    corpus_analysis[
        ["doc_id", "source_type", "publication_year", "clean_text"]
    ]
    .assign(
        token=lambda x:
        x["clean_text"].str.lower().str.findall(token_pattern)
    )
    .explode("token")
    .dropna(subset=["token"])
    .drop(columns="clean_text")
    .reset_index(drop=True)
)

print("Total tokens:", len(tokens_unfiltered))
print("Unique tokens:", tokens_unfiltered["token"].nunique())

##removing stopwords

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = {"et", "al"}

tokens_filtered = tokens_unfiltered[
    ~tokens_unfiltered["token"].isin(ENGLISH_STOP_WORDS)
    & ~tokens_unfiltered["token"].isin(custom_stopwords)
    & tokens_unfiltered["token"].str.contains(r"[a-z]", regex=True)
].copy()

filtered_top_tokens = (
    tokens_filtered["token"]
    .value_counts()
    .head(40)
    .rename_axis("token")
    .reset_index(name="frequency")
)

print("Tokens before filtering:", len(tokens_unfiltered))
print("Tokens after filtering:", len(tokens_filtered))

filtered_top_tokens

Total tokens: 94940
Unique tokens: 7371
Tokens before filtering: 94940
Tokens after filtering: 55557


,token,frequency
0,heat,1165
1,health,792
2,children,720
3,climate,671
4,bangladesh,575
5,extreme,487
6,change,412
7,temperatures,405
8,water,289
9,exposure,251


In [11]:
## TF-IDF 2
##

number_of_documents = corpus_analysis["doc_id"].nunique()

token_statistics = (
    tokens_filtered
    .groupby("token")
    .agg(
        frequency=("token", "size"),
        document_frequency=("doc_id", "nunique")
    )
    .reset_index()
)

token_statistics["document_percentage"] = (
    token_statistics["document_frequency"]
    / number_of_documents
    * 100
).round(1)

token_statistics = token_statistics.sort_values(
    ["document_frequency", "frequency"],
    ascending=False
).reset_index(drop=True)

token_statistics.head(40)

,token,frequency,document_frequency,document_percentage
0,heat,1165,69,95.8
1,bangladesh,575,68,94.4
2,climate,671,67,93.1
3,children,720,65,90.3
4,health,792,64,88.9
5,temperatures,405,62,86.1
6,change,412,60,83.3
7,vulnerable,182,59,81.9
8,extreme,487,58,80.6
9,risk,161,54,75.0


In [12]:
##Lemmatization

%pip install -q spacy

import sys
import subprocess
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "spacy",
        "download",
        "en_core_web_sm"
    ])
    nlp = spacy.load("en_core_web_sm")

# Keep the components required for accurate lemmatization
nlp.disable_pipes("parser", "ner")


Note: you may need to restart the kernel to use updated packages.


['parser', 'ner']

In [13]:
import sys

print("Python:", sys.executable)
print("spaCy version:", spacy.__version__)

!{sys.executable} -m spacy download en_core_web_sm

Python: C:\Users\mishu\anaconda3\python.exe
spaCy version: 3.8.16


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mishu\anaconda3\Lib\site-packages\spacy\__main__.py", line 4, in <module>
    setup_cli()
    ~~~~~~~~~^^
  File "C:\Users\mishu\anaconda3\Lib\site-packages\spacy\cli\_util.py", line 77, in setup_cli
    command = get_command(app)
  File "C:\Users\mishu\anaconda3\Lib\site-packages\typer\main.py", line 350, in get_command
    click_command: click.Command = get_group(typer_instance)
                                   ~~~~~~~~~^^^^^^^^^^^^^^^^
  File "C:\Users\mishu\anaconda3\Lib\site-packages\typer\main.py", line 332, in get_group
    group = get_group_from_info(
        TyperInfo(typer_instance),
        pretty_exceptions_short=typer_instance.pretty_exceptions_short,
        rich_markup_mode=typer_instance.rich_markup_mode,
    )
  File "C:\Users\mishu\anaconda3\Lib\site-packages\typer\main.py", line 483, in get_group_from_info
    

In [14]:
import importlib
import spacy
from spacy.cli.download import download

# Download the model without using spaCy's broken command-line interface
download("en_core_web_sm")

# Make the newly installed package visible
importlib.invalidate_caches()

# Load the model
nlp = spacy.load("en_core_web_sm")

# These components are unnecessary for lemmatization
nlp.disable_pipes("parser", "ner")

print("spaCy model loaded successfully.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy model loaded successfully.


In [15]:
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = {"et", "al"}

document_columns = [
    "doc_id",
    "source_type",
    "publication_year",
    "clean_text"
]

documents_for_lemmatization = corpus_analysis[
    document_columns
].copy()

documents_for_lemmatization["clean_text"] = (
    documents_for_lemmatization["clean_text"]
    .fillna("")
    .astype(str)
)

lemmatized_records = []

texts = documents_for_lemmatization["clean_text"].tolist()

for document_row, processed_document in zip(
    documents_for_lemmatization.itertuples(index=False),
    nlp.pipe(texts, batch_size=8)
):
    for word in processed_document:

        original_token = word.text.lower().strip()
        lemma = word.lemma_.lower().strip()

        # Remove spaces and punctuation
        if word.is_space or word.is_punct:
            continue

        # Remove standard and custom stopwords
        if original_token in ENGLISH_STOP_WORDS:
            continue

        if lemma in ENGLISH_STOP_WORDS:
            continue

        if original_token in custom_stopwords:
            continue

        if lemma in custom_stopwords:
            continue

        # Remove numbers and symbols without letters
        if not re.search(r"[a-z]", lemma):
            continue

        lemmatized_records.append({
            "doc_id": document_row.doc_id,
            "source_type": document_row.source_type,
            "publication_year": document_row.publication_year,
            "original_token": original_token,
            "lemma": lemma
        })

tokens_lemmatized = pd.DataFrame(lemmatized_records)

print("Included documents:",
      tokens_lemmatized["doc_id"].nunique())

print("Total lemmatized tokens:",
      len(tokens_lemmatized))

print("Unique original tokens:",
      tokens_lemmatized["original_token"].nunique())

print("Unique lemmas:",
      tokens_lemmatized["lemma"].nunique())

Included documents: 72
Total lemmatized tokens: 56408
Unique original tokens: 6382
Unique lemmas: 5002


In [16]:
lemma_examples = (
    tokens_lemmatized[
        tokens_lemmatized["original_token"]
        != tokens_lemmatized["lemma"]
    ][["original_token", "lemma"]]
    .drop_duplicates()
    .head(40)
)

lemma_examples

,original_token,lemma
2,exposing,expose
3,millions,million
4,children,child
12,aged,age
16,experiencing,experience
26,according,accord
29,led,lead
34,published,publish
41,tripled,triple
48,months,month


In [17]:
number_of_documents = corpus_analysis["doc_id"].nunique()

lemma_statistics = (
    tokens_lemmatized
    .groupby("lemma")
    .agg(
        frequency=("lemma", "size"),
        document_frequency=("doc_id", "nunique")
    )
    .reset_index()
)

lemma_statistics["document_percentage"] = (
    lemma_statistics["document_frequency"]
    / number_of_documents
    * 100
).round(1)

lemma_statistics = lemma_statistics.sort_values(
    ["document_frequency", "frequency"],
    ascending=False
).reset_index(drop=True)

print("Documents:", number_of_documents)
print("Total lemmatized tokens:", len(tokens_lemmatized))
print("Unique lemmas:", tokens_lemmatized["lemma"].nunique())

lemma_statistics.head(50)

Documents: 72
Total lemmatized tokens: 56408
Unique lemmas: 5002


,lemma,frequency,document_frequency,document_percentage
0,heat,1364,69,95.8
1,temperature,647,69,95.8
2,bangladesh,635,69,95.8
3,climate,777,68,94.4
4,child,1029,67,93.1
5,health,805,64,88.9
6,change,509,63,87.5
7,high,299,62,86.1
8,country,242,61,84.7
9,day,289,60,83.3


In [18]:
from pathlib import Path

output_folder = Path("../outputs")
output_folder.mkdir(exist_ok=True)

tokens_lemmatized.to_csv(
    output_folder / "tokens_lemmatized.csv",
    index=False,
    encoding="utf-8-sig"
)

lemma_statistics.to_csv(
    output_folder / "lemma_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)


In [19]:
short_lemma_review = (
    tokens_lemmatized[
        tokens_lemmatized["lemma"].str.len() <= 2
    ]
    .groupby(["lemma", "original_token"])
    .agg(
        frequency=("lemma", "size"),
        document_frequency=("doc_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["document_frequency", "frequency"],
        ascending=False
    )
    .reset_index(drop=True)
)

short_lemma_review.head(50)

,lemma,original_token,frequency,document_frequency
0,’s,’s,363,48
1,'s,'s,150,32
2,c,c,199,29
3,dr,dr,23,10
4,f,f,11,6
5,km,km,11,4
6,el,el,9,4
7,p,p,19,3
8,ci,ci,24,2
9,b,b,10,2


In [20]:
lemma_text = tokens_lemmatized["lemma"].astype(str)

technical_noise_mask = (
    # Possessive fragments and titles
    lemma_text.isin({"’s", "'s", "dr"})
    
    # Any isolated letter
    | lemma_text.str.fullmatch(r"[a-z]", na=False)
    
    # Letter followed by a period, such as c. or q.
    | lemma_text.str.fullmatch(r"[a-z]\.", na=False)
    
    # Table labels such as a1, a4 or a9
    | lemma_text.str.fullmatch(r"[a-z]\d+", na=False)
    
    # Malformed fragments such as 2c or 5c
    | lemma_text.str.fullmatch(r"\d+[a-z]", na=False)
)

removed_technical_noise = (
    tokens_lemmatized.loc[technical_noise_mask, "lemma"]
    .value_counts()
    .rename_axis("removed_lemma")
    .reset_index(name="frequency")
)

tokens_base_clean = (
    tokens_lemmatized.loc[~technical_noise_mask]
    .copy()
    .reset_index(drop=True)
)

print("Tokens before:", len(tokens_lemmatized))
print("Tokens after:", len(tokens_base_clean))
print("Tokens removed:", technical_noise_mask.sum())

removed_technical_noise

Tokens before: 56408
Tokens after: 55538
Tokens removed: 870


,removed_lemma,frequency
0,’s,363
1,c,199
2,'s,150
3,dr,23
4,p,19
...,...,...
62,h7,1
63,h8,1
64,h9,1
65,1990s,1


In [21]:
tokens_base_clean.to_csv(
    "../outputs/tokens_base_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: ../outputs/tokens_base_clean.csv")

Saved: ../outputs/tokens_base_clean.csv


In [22]:
# Convert the lemma column to text
lemma_text = tokens_lemmatized["lemma"].astype(str)

# Identify clear technical noise
technical_noise_mask = (
    # Possessive fragments and titles
    lemma_text.isin({"’s", "'s", "dr"})
    
    # Isolated letters such as c, f, p and r
    | lemma_text.str.fullmatch(r"[a-z]", na=False)
    
    # Letter followed by a period, such as c. or q.
    | lemma_text.str.fullmatch(r"[a-z]\.", na=False)
    
    # Table labels such as a1, a4 and h7
    | lemma_text.str.fullmatch(r"[a-z]\d+", na=False)
    
    # Temperature fragments such as 2c, 10c and 95f
    | lemma_text.str.fullmatch(r"\d{1,3}[cf]", na=False)
)

# Create an audit table of removed lemmas
removed_technical_noise = (
    tokens_lemmatized.loc[
        technical_noise_mask,
        "lemma"
    ]
    .value_counts()
    .rename_axis("removed_lemma")
    .reset_index(name="frequency")
)

# Create the cleaned token table
tokens_base_clean = (
    tokens_lemmatized.loc[
        ~technical_noise_mask
    ]
    .copy()
    .reset_index(drop=True)
)

# Count decade expressions before cleaning
decades_before = (
    tokens_lemmatized.loc[
        tokens_lemmatized["lemma"].str.fullmatch(
            r"\d{4}s",
            na=False
        ),
        "lemma"
    ]
    .value_counts()
    .rename("frequency_before")
)

# Count decade expressions after cleaning
decades_after = (
    tokens_base_clean.loc[
        tokens_base_clean["lemma"].str.fullmatch(
            r"\d{4}s",
            na=False
        ),
        "lemma"
    ]
    .value_counts()
    .rename("frequency_after")
)

# Compare decade counts before and after cleaning
decade_validation = (
    pd.concat(
        [decades_before, decades_after],
        axis=1
    )
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"lemma": "decade"})
)

decade_validation["all_occurrences_retained"] = (
    decade_validation["frequency_before"]
    == decade_validation["frequency_after"]
)

# Display validation results
print("Tokens before:", len(tokens_lemmatized))
print("Tokens after:", len(tokens_base_clean))
print("Tokens removed:", technical_noise_mask.sum())

print("\nRemoved technical noise:")
display(removed_technical_noise)

print("\nDecade validation:")
display(decade_validation)

# Save the corrected token table
tokens_base_clean.to_csv(
    "../outputs/tokens_base_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: ../outputs/tokens_base_clean.csv")

Tokens before: 56408
Tokens after: 55550
Tokens removed: 858

Removed technical noise:


,removed_lemma,frequency
0,’s,363
1,c,199
2,'s,150
3,dr,23
4,p,19
5,f,11
6,b,10
7,z,8
8,40c,6
9,42c,4



Decade validation:


,decade,frequency_before,frequency_after,all_occurrences_retained
0,1950s,2,2,True
1,1980s,1,1,True
2,1970s,1,1,True
3,1990s,1,1,True



Saved: ../outputs/tokens_base_clean.csv


In [23]:
thematic_stopwords = {
    "accord",        # produced from "according"
    "say",           # reporting verb: said, says
    "include",       # generic listing word
    "like",          # generic comparison word
    "particularly",  # generic emphasis word
    "relate"         # often produced from "heat-related"
}

tokens_thematic = (
    tokens_base_clean[
        ~tokens_base_clean["lemma"].isin(thematic_stopwords)
    ]
    .copy()
    .reset_index(drop=True)
)

removed_thematic_words = (
    tokens_base_clean[
        tokens_base_clean["lemma"].isin(thematic_stopwords)
    ]["lemma"]
    .value_counts()
    .rename_axis("removed_lemma")
    .reset_index(name="frequency")
)

print("Base-clean tokens:", len(tokens_base_clean))
print("Thematic tokens:", len(tokens_thematic))
print(
    "Generic thematic words removed:",
    len(tokens_base_clean) - len(tokens_thematic)
)

removed_thematic_words

Base-clean tokens: 55550
Thematic tokens: 54627
Generic thematic words removed: 923


,removed_lemma,frequency
0,relate,226
1,say,199
2,include,160
3,particularly,145
4,like,121
5,accord,72


In [26]:
# Save the thematic token dataset
tokens_thematic.to_csv(
    "../outputs/tokens_thematic.csv",
    index=False,
    encoding="utf-8-sig"
)

# Document the custom stopwords and reasons
thematic_stopword_log = pd.DataFrame({
    "stopword": [
        "accord",
        "say",
        "include",
        "like",
        "particularly",
        "relate"
    ],
    "reason": [
        "Lemma generated from the reporting phrase 'according to'",
        "Generic reporting verb",
        "Generic listing verb",
        "Generic comparison word",
        "Generic emphasis word",
        "Usually generated from the compound 'heat-related'"
    ]
})

thematic_stopword_log.to_csv(
    "../outputs/thematic_stopword_log.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Thematic tokens saved:",
      Path("../outputs/tokens_thematic.csv").exists())

print("Stopword log saved:",
      Path("../outputs/thematic_stopword_log.csv").exists())

print("Final thematic-token count:", len(tokens_thematic))

Thematic tokens saved: True
Stopword log saved: True
Final thematic-token count: 54627


In [27]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Reconstruct one lemmatized text per document
document_lemmas = (
    tokens_thematic
    .groupby("doc_id", sort=False)["lemma"]
    .agg(" ".join)
)

# Create two-word phrases appearing in at least 3 documents
bigram_vectorizer = CountVectorizer(
    ngram_range=(2, 2),
    min_df=3
)

bigram_matrix = bigram_vectorizer.fit_transform(
    document_lemmas
)

bigram_names = bigram_vectorizer.get_feature_names_out()

# Total occurrences across the corpus
bigram_frequency = np.asarray(
    bigram_matrix.sum(axis=0)
).ravel()

# Number of documents containing each phrase
bigram_document_frequency = np.asarray(
    (bigram_matrix > 0).sum(axis=0)
).ravel()

bigram_statistics = pd.DataFrame({
    "bigram": bigram_names,
    "frequency": bigram_frequency,
    "document_frequency": bigram_document_frequency
})

bigram_statistics["document_percentage"] = (
    bigram_statistics["document_frequency"]
    / len(document_lemmas)
    * 100
).round(1)

bigram_statistics = (
    bigram_statistics
    .sort_values(
        ["document_frequency", "frequency"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Documents:", len(document_lemmas))
print("Candidate bigrams:", len(bigram_statistics))

bigram_statistics.head(50)

Documents: 72
Candidate bigrams: 2099


,bigram,frequency,document_frequency,document_percentage
0,climate change,407,61,84.7
1,extreme heat,376,45,62.5
2,heat stress,152,38,52.8
3,rise temperature,100,32,44.4
4,heat illness,71,31,43.1
5,impact climate,51,31,43.1
6,million child,61,29,40.3
7,high temperature,73,27,37.5
8,pregnant woman,56,27,37.5
9,heat wave,53,27,37.5


In [28]:
##bigram association score

from collections import Counter
import numpy as np
import pandas as pd

# Count all individual lemmas
unigram_counts = Counter(tokens_thematic["lemma"])

# Total numbers used in the PMI calculation
total_unigrams = len(tokens_thematic)
total_bigrams = int(bigram_statistics["frequency"].sum())

# Separate the two words in each bigram
bigram_scored = bigram_statistics.copy()

bigram_scored[["word_1", "word_2"]] = (
    bigram_scored["bigram"]
    .str.split(" ", n=1, expand=True)
)

# Add individual-word frequencies
bigram_scored["word_1_frequency"] = (
    bigram_scored["word_1"].map(unigram_counts)
)

bigram_scored["word_2_frequency"] = (
    bigram_scored["word_2"].map(unigram_counts)
)

# Calculate pointwise mutual information
bigram_scored["pmi"] = np.log2(
    (
        bigram_scored["frequency"] * total_unigrams**2
    )
    /
    (
        total_bigrams
        * bigram_scored["word_1_frequency"]
        * bigram_scored["word_2_frequency"]
    )
)

# Require enough evidence to avoid emphasizing rare coincidences
bigram_review = (
    bigram_scored[
        (bigram_scored["frequency"] >= 10)
        & (bigram_scored["document_frequency"] >= 5)
    ]
    .sort_values(
        ["pmi", "document_frequency", "frequency"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Bigrams retained for review:", len(bigram_review))

bigram_review[
    [
        "bigram",
        "frequency",
        "document_frequency",
        "document_percentage",
        "pmi"
    ]
].head(50)

Bigrams retained for review: 234


,bigram,frequency,document_frequency,document_percentage,pmi
0,sheldon yett,11,11,15.3,14.070010
1,united nations,13,12,16.7,13.847618
2,densely populate,12,7,9.7,13.147178
3,data collection,11,5,6.9,12.914732
4,tell afp,12,6,8.3,12.839056
5,index ccri,11,10,13.9,12.654973
6,directorate general,10,9,12.5,12.391939
7,greenhouse gas,15,9,12.5,12.369571
8,bear brunt,11,11,15.3,12.171890
9,meteorological department,15,11,15.3,12.160984


In [29]:
import re
import pandas as pd

phrases_to_inspect = [
    "sheldon yett",
    "tell afp",
    "index ccri",
    "unicef urge",
    "primary mass",
    "mass education"
]

phrase_document_rows = []

for phrase in phrases_to_inspect:
    phrase_pattern = rf"\b{re.escape(phrase)}\b"

    matching_documents = document_lemmas[
        document_lemmas.str.contains(
            phrase_pattern,
            regex=True,
            na=False
        )
    ].index

    for doc_id in matching_documents:
        phrase_document_rows.append({
            "phrase": phrase,
            "doc_id": doc_id
        })

phrase_document_review = pd.DataFrame(
    phrase_document_rows
)

# Include whichever metadata columns are available
metadata_candidates = [
    "doc_id",
    "title",
    "source_type",
    "publication_date",
    "year",
    "domain",
    "url"
]

metadata_columns = [
    column
    for column in metadata_candidates
    if column in corpus_analysis.columns
]

phrase_document_review = (
    phrase_document_review
    .merge(
        corpus_analysis[metadata_columns].drop_duplicates(
            subset="doc_id"
        ),
        on="doc_id",
        how="left"
    )
    .sort_values(["phrase", "doc_id"])
    .reset_index(drop=True)
)

phrase_document_review

,phrase,doc_id,source_type,url
0,index ccri,DOC_002,civil society,https://www.unicef.org/bangladesh/en/press-rel...
1,index ccri,DOC_008,news paper,https://www.thedailystar.net/news/bangladesh/n...
2,index ccri,DOC_011,news paper,https://en.prothomalo.com/bangladesh/wftdj08h8e
3,index ccri,DOC_012,news paper,https://www.pressenza.com/2025/07/children-at-...
4,index ccri,DOC_034,news paper,https://english.news.cn/20240424/55dd98f670c54...
5,index ccri,DOC_037,news paper,https://www.risingbd.com/english/national/news...
6,index ccri,DOC_042,news paper,https://voice7news.tv/public/bangladesh/news/3834
7,index ccri,DOC_052,news paper,https://viewsbangladesh.com/children-of-bangla...
8,index ccri,DOC_054,news paper,https://www.thedailystar.net/star-health/news/...
9,index ccri,DOC_083,civil society,https://www.unicef.org/press-releases/soaring-...


In [30]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Keep document order stable
similarity_documents = document_lemmas.sort_index()
document_ids = similarity_documents.index.to_list()

# Use sequences of 3–5 words to detect shared wording
overlap_vectorizer = TfidfVectorizer(
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True
)

overlap_matrix = overlap_vectorizer.fit_transform(
    similarity_documents
)

similarity_matrix = cosine_similarity(overlap_matrix)

# Convert the upper triangle into document pairs
similarity_rows = []

for first_index in range(len(document_ids)):
    for second_index in range(
        first_index + 1,
        len(document_ids)
    ):
        similarity_rows.append({
            "doc_id_1": document_ids[first_index],
            "doc_id_2": document_ids[second_index],
            "cosine_similarity": similarity_matrix[
                first_index,
                second_index
            ]
        })

document_similarity_pairs = (
    pd.DataFrame(similarity_rows)
    .sort_values(
        "cosine_similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

# Add available document metadata
metadata_candidates = [
    "doc_id",
    "title",
    "source_type",
    "url"
]

metadata_columns = [
    column
    for column in metadata_candidates
    if column in corpus_analysis.columns
]

document_metadata = (
    corpus_analysis[metadata_columns]
    .drop_duplicates(subset="doc_id")
)

document_similarity_review = (
    document_similarity_pairs
    .merge(
        document_metadata.add_suffix("_1"),
        on="doc_id_1",
        how="left"
    )
    .merge(
        document_metadata.add_suffix("_2"),
        on="doc_id_2",
        how="left"
    )
)

document_similarity_review[
    [
        column
        for column in [
            "doc_id_1",
            "doc_id_2",
            "cosine_similarity",
            "title_1",
            "title_2",
            "source_type_1",
            "source_type_2"
        ]
        if column in document_similarity_review.columns
    ]
].head(30)

,doc_id_1,doc_id_2,cosine_similarity,source_type_1,source_type_2
0,DOC_007,DOC_068,0.985480,civil society,news paper
1,DOC_043,DOC_068,0.948256,civil society,news paper
2,DOC_007,DOC_043,0.934372,civil society,civil society
3,DOC_019,DOC_023,0.902858,civil society,news paper
4,DOC_002,DOC_052,0.857464,civil society,news paper
5,DOC_013,DOC_025,0.854975,civil society,news paper
6,DOC_002,DOC_011,0.800920,civil society,news paper
7,DOC_011,DOC_052,0.789797,news paper,news paper
8,DOC_008,DOC_011,0.728154,news paper,news paper
9,DOC_002,DOC_008,0.712332,civil society,news paper


In [31]:
# Provisional threshold for manual near-duplicate review
similarity_threshold = 0.70

near_duplicate_pairs = (
    document_similarity_pairs[
        document_similarity_pairs["cosine_similarity"]
        >= similarity_threshold
    ]
    .copy()
    .reset_index(drop=True)
)

# Union–find functions for grouping connected document pairs
parent = {
    doc_id: doc_id
    for doc_id in document_ids
}

def find_root(doc_id):
    while parent[doc_id] != doc_id:
        parent[doc_id] = parent[parent[doc_id]]
        doc_id = parent[doc_id]
    return doc_id

def join_documents(doc_id_1, doc_id_2):
    root_1 = find_root(doc_id_1)
    root_2 = find_root(doc_id_2)

    if root_1 != root_2:
        parent[root_2] = root_1

# Join documents connected by high similarity
for _, row in near_duplicate_pairs.iterrows():
    join_documents(
        row["doc_id_1"],
        row["doc_id_2"]
    )

# Collect connected groups
connected_groups = {}

for doc_id in document_ids:
    root = find_root(doc_id)

    connected_groups.setdefault(
        root,
        []
    ).append(doc_id)

# Retain only groups containing multiple documents
duplicate_groups = [
    sorted(group)
    for group in connected_groups.values()
    if len(group) > 1
]

duplicate_groups = sorted(
    duplicate_groups,
    key=lambda group: group[0]
)

# Create a document-level review table
cluster_rows = []

for cluster_number, group in enumerate(
    duplicate_groups,
    start=1
):
    cluster_id = f"ND_{cluster_number:02d}"

    for doc_id in group:
        cluster_rows.append({
            "doc_id": doc_id,
            "near_duplicate_cluster": cluster_id,
            "thematic_token_count": len(
                document_lemmas.loc[doc_id].split()
            )
        })

near_duplicate_clusters = pd.DataFrame(
    cluster_rows
)

# Add available metadata
near_duplicate_cluster_review = (
    near_duplicate_clusters
    .merge(
        document_metadata,
        on="doc_id",
        how="left"
    )
    .sort_values(
        ["near_duplicate_cluster", "doc_id"]
    )
    .reset_index(drop=True)
)

near_duplicate_cluster_review

,doc_id,near_duplicate_cluster,thematic_token_count,source_type,url
0,DOC_002,ND_01,217,civil society,https://www.unicef.org/bangladesh/en/press-rel...
1,DOC_008,ND_01,222,news paper,https://www.thedailystar.net/news/bangladesh/n...
2,DOC_011,ND_01,227,news paper,https://en.prothomalo.com/bangladesh/wftdj08h8e
3,DOC_052,ND_01,224,news paper,https://viewsbangladesh.com/children-of-bangla...
4,DOC_007,ND_02,350,civil society,https://www.preventionweb.net/news/new-guideli...
5,DOC_043,ND_02,372,civil society,https://bangladesh.un.org/en/267869-unicef-sup...
6,DOC_068,ND_02,350,news paper,https://reliefweb.int/report/bangladesh/new-gu...
7,DOC_013,ND_03,342,civil society,https://www.unicef.org/bangladesh/en/press-rel...
8,DOC_025,ND_03,332,news paper,https://bdnews24.com/bangladesh/7387478f9b08
9,DOC_019,ND_04,349,civil society,https://www.savethechildren.org/us/about-us/me...


In [32]:
# Select one representative from each confirmed cluster
cluster_representatives = {
    "ND_01": "DOC_002",
    "ND_02": "DOC_043",
    "ND_03": "DOC_013",
    "ND_04": "DOC_019"
}

# Record the representative selected for each cluster
near_duplicate_decisions = (
    near_duplicate_cluster_review
    .copy()
)

near_duplicate_decisions["representative_doc_id"] = (
    near_duplicate_decisions[
        "near_duplicate_cluster"
    ].map(cluster_representatives)
)

near_duplicate_decisions[
    "retain_for_unique_content_analysis"
] = (
    near_duplicate_decisions["doc_id"]
    == near_duplicate_decisions["representative_doc_id"]
)

near_duplicate_decisions["decision_reason"] = np.where(
    near_duplicate_decisions[
        "retain_for_unique_content_analysis"
    ],
    "Retained as cluster representative",
    "Retained in master corpus but omitted from unique-content analysis"
)

# Identify the seven non-representative documents
near_duplicate_omissions = set(
    near_duplicate_decisions.loc[
        ~near_duplicate_decisions[
            "retain_for_unique_content_analysis"
        ],
        "doc_id"
    ]
)

print(
    "Documents omitted from unique-content analysis:",
    sorted(near_duplicate_omissions)
)

# Add duplicate information to the 72-document corpus
cluster_information = (
    near_duplicate_decisions[
        [
            "doc_id",
            "near_duplicate_cluster",
            "representative_doc_id"
        ]
    ]
    .drop_duplicates(subset="doc_id")
)

corpus_analysis_flagged = (
    corpus_analysis
    .merge(
        cluster_information,
        on="doc_id",
        how="left"
    )
)

corpus_analysis_flagged[
    "retain_for_unique_content_analysis"
] = (
    ~corpus_analysis_flagged["doc_id"].isin(
        near_duplicate_omissions
    )
)

# Create the deduplicated analytical corpus
corpus_analysis_unique = (
    corpus_analysis_flagged[
        corpus_analysis_flagged[
            "retain_for_unique_content_analysis"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Create corresponding thematic-token dataset
tokens_thematic_unique = (
    tokens_thematic[
        tokens_thematic["doc_id"].isin(
            corpus_analysis_unique["doc_id"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Reconstruct document texts from the unique-content tokens
document_lemmas_unique = (
    tokens_thematic_unique
    .groupby("doc_id", sort=False)["lemma"]
    .agg(" ".join)
)

print("\nMaster analytical corpus:", len(corpus_analysis_flagged))
print("Unique-content corpus:", len(corpus_analysis_unique))
print("Documents omitted as near-duplicates:",
      len(near_duplicate_omissions))

print("\nUnique-content source distribution:")
print(
    corpus_analysis_unique["source_type"]
    .value_counts()
)

# Save the audit files
near_duplicate_decisions.to_csv(
    "../outputs/near_duplicate_decisions.csv",
    index=False,
    encoding="utf-8-sig"
)

corpus_analysis_flagged.to_csv(
    "../outputs/corpus_analysis_flagged.csv",
    index=False,
    encoding="utf-8-sig"
)

corpus_analysis_unique.to_csv(
    "../outputs/corpus_analysis_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

tokens_thematic_unique.to_csv(
    "../outputs/tokens_thematic_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

Documents omitted from unique-content analysis: ['DOC_007', 'DOC_008', 'DOC_011', 'DOC_023', 'DOC_025', 'DOC_052', 'DOC_068']

Master analytical corpus: 72
Unique-content corpus: 65
Documents omitted as near-duplicates: 7

Unique-content source distribution:
source_type
news paper       40
civil society    15
scientific        9
others            1
Name: count, dtype: int64


In [33]:
# Inspect the document classified as "others"
metadata_candidates = [
    "doc_id",
    "title",
    "source_type",
    "publisher",
    "domain",
    "url"
]

available_metadata = [
    column
    for column in metadata_candidates
    if column in corpus_analysis_unique.columns
]

other_source_review = corpus_analysis_unique.loc[
    corpus_analysis_unique["source_type"]
    .str.strip()
    .str.lower()
    .eq("others"),
    available_metadata
]

other_source_review.T

,42
doc_id,DOC_057
source_type,others
url,https://nationalhospital.com.bd/extreme-heat-i...


In [34]:
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer

# --------------------------------------------------
# 1. Divide the unique-content corpus
# --------------------------------------------------

public_source_types = {
    "news paper",
    "civil society",
    "others"
}

normalized_source_type = (
    corpus_analysis_unique["source_type"]
    .str.strip()
    .str.lower()
)

public_corpus = (
    corpus_analysis_unique[
        normalized_source_type.isin(public_source_types)
    ]
    .copy()
    .reset_index(drop=True)
)

scientific_corpus = (
    corpus_analysis_unique[
        normalized_source_type.eq("scientific")
    ]
    .copy()
    .reset_index(drop=True)
)

# --------------------------------------------------
# 2. Create corresponding token datasets
# --------------------------------------------------

public_document_ids = set(public_corpus["doc_id"])
scientific_document_ids = set(scientific_corpus["doc_id"])

tokens_thematic_public = (
    tokens_thematic_unique[
        tokens_thematic_unique["doc_id"].isin(
            public_document_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

tokens_thematic_scientific = (
    tokens_thematic_unique[
        tokens_thematic_unique["doc_id"].isin(
            scientific_document_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Reconstruct one processed text per public document
document_lemmas_public = (
    tokens_thematic_public
    .groupby("doc_id", sort=False)["lemma"]
    .agg(" ".join)
)

# --------------------------------------------------
# 3. Calculate public-corpus unigram statistics
# --------------------------------------------------

number_of_public_documents = len(public_corpus)

public_unigram_statistics = (
    tokens_thematic_public
    .groupby("lemma")
    .agg(
        frequency=("lemma", "size"),
        document_frequency=("doc_id", "nunique")
    )
    .reset_index()
)

public_unigram_statistics["document_percentage"] = (
    public_unigram_statistics["document_frequency"]
    / number_of_public_documents
    * 100
).round(1)

public_unigram_statistics = (
    public_unigram_statistics
    .sort_values(
        ["document_frequency", "frequency"],
        ascending=False
    )
    .reset_index(drop=True)
)

# --------------------------------------------------
# 4. Calculate public-corpus bigram statistics
# --------------------------------------------------

public_bigram_vectorizer = CountVectorizer(
    ngram_range=(2, 2),
    min_df=3
)

public_bigram_matrix = (
    public_bigram_vectorizer
    .fit_transform(document_lemmas_public)
)

public_bigram_names = (
    public_bigram_vectorizer
    .get_feature_names_out()
)

public_bigram_frequency = np.asarray(
    public_bigram_matrix.sum(axis=0)
).ravel()

public_bigram_document_frequency = np.asarray(
    (public_bigram_matrix > 0).sum(axis=0)
).ravel()

public_bigram_statistics = pd.DataFrame({
    "bigram": public_bigram_names,
    "frequency": public_bigram_frequency,
    "document_frequency":
        public_bigram_document_frequency
})

public_bigram_statistics["document_percentage"] = (
    public_bigram_statistics["document_frequency"]
    / number_of_public_documents
    * 100
).round(1)

# --------------------------------------------------
# 5. Calculate corrected PMI scores
# --------------------------------------------------

public_unigram_counts = Counter(
    tokens_thematic_public["lemma"]
)

total_public_unigrams = len(tokens_thematic_public)

# All possible bigram positions, including rare bigrams
total_public_bigram_positions = sum(
    max(len(text.split()) - 1, 0)
    for text in document_lemmas_public
)

public_bigram_statistics[
    ["word_1", "word_2"]
] = (
    public_bigram_statistics["bigram"]
    .str.split(" ", n=1, expand=True)
)

public_bigram_statistics["word_1_frequency"] = (
    public_bigram_statistics["word_1"]
    .map(public_unigram_counts)
)

public_bigram_statistics["word_2_frequency"] = (
    public_bigram_statistics["word_2"]
    .map(public_unigram_counts)
)

public_bigram_statistics["pmi"] = np.log2(
    (
        public_bigram_statistics["frequency"]
        / total_public_bigram_positions
    )
    /
    (
        (
            public_bigram_statistics["word_1_frequency"]
            / total_public_unigrams
        )
        *
        (
            public_bigram_statistics["word_2_frequency"]
            / total_public_unigrams
        )
    )
)

public_bigram_statistics = (
    public_bigram_statistics
    .sort_values(
        ["document_frequency", "frequency"],
        ascending=False
    )
    .reset_index(drop=True)
)

public_bigram_review = (
    public_bigram_statistics[
        (public_bigram_statistics["frequency"] >= 10)
        & (
            public_bigram_statistics[
                "document_frequency"
            ] >= 5
        )
    ]
    .sort_values(
        ["pmi", "document_frequency", "frequency"],
        ascending=False
    )
    .reset_index(drop=True)
)

# --------------------------------------------------
# 6. Validate and save everything
# --------------------------------------------------

print("Public-facing documents:", len(public_corpus))
print("Scientific documents:", len(scientific_corpus))

print("\nPublic source distribution:")
print(public_corpus["source_type"].value_counts())

print("\nPublic thematic tokens:",
      len(tokens_thematic_public))

print("Candidate public bigrams:",
      len(public_bigram_statistics))

print("Public bigrams retained for review:",
      len(public_bigram_review))

public_corpus.to_csv(
    "../outputs/public_corpus_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

scientific_corpus.to_csv(
    "../outputs/scientific_corpus_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

tokens_thematic_public.to_csv(
    "../outputs/tokens_thematic_public.csv",
    index=False,
    encoding="utf-8-sig"
)

public_unigram_statistics.to_csv(
    "../outputs/public_unigram_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)

public_bigram_statistics.to_csv(
    "../outputs/public_bigram_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)

public_bigram_review.to_csv(
    "../outputs/public_bigram_review.csv",
    index=False,
    encoding="utf-8-sig"
)

display(public_unigram_statistics.head(30))

display(
    public_bigram_statistics[
        [
            "bigram",
            "frequency",
            "document_frequency",
            "document_percentage",
            "pmi"
        ]
    ].head(30)
)

Public-facing documents: 56
Scientific documents: 9

Public source distribution:
source_type
news paper       40
civil society    15
others            1
Name: count, dtype: int64

Public thematic tokens: 21579
Candidate public bigrams: 831
Public bigrams retained for review: 67


,lemma,frequency,document_frequency,document_percentage
0,heat,592,54,96.4
1,bangladesh,312,53,94.6
2,temperature,266,53,94.6
3,climate,313,52,92.9
4,child,423,51,91.1
5,health,282,48,85.7
6,country,139,48,85.7
7,change,160,47,83.9
8,high,121,47,83.9
9,heatwave,223,46,82.1


,bigram,frequency,document_frequency,document_percentage,pmi
0,climate change,144,45,80.4,5.959067
1,extreme heat,126,37,66.1,4.591487
2,heat stress,68,26,46.4,4.869708
3,rise temperature,42,24,42.9,4.807387
4,million child,47,23,41.1,4.489697
5,heat illness,31,22,39.3,4.263189
6,heat wave,37,19,33.9,5.153162
7,public health,30,19,33.9,5.553719
8,high temperature,29,19,33.9,4.284925
9,pregnant woman,36,18,32.1,7.981831
